# Experiment 2 — what suppression costs, and what specialisation buys

Matched pairs: each general checkpoint against its math-tuned counterpart of the same generation,
across four prompt conditions. Two questions:

1. **What does suppression cost a general model?** Forced blurt against told-to-reason, same model,
   same items.
2. **What does a math-tuned model buy?** A 1.5B math model against a 7B general one, measured both
   when the 7B is forbidden to reason and when it is asked to.

Experiment 1 (`experiment_1_which_checkpoints_blurt.ipynb`) established that general models comply with a blurt
instruction and lose 40-75 points, while most math-tuned models ignore it entirely. This sizes
that effect across matched pairs. Writes `out/experiment_2/`.

## How to run this

| conditions | models | datasets | items | generations |
|---|---|---|---|---|
| 4 | 9 | 2 | 20 | **1,440** |

Roughly 4-5 hours on a 16 GB GPU; the four 7B checkpoints run at 8-bit and dominate. Resumable —
every generation is keyed by `RUN_ID` in `traces.jsonl`, so an interrupted run continues.

To halve it, set `DATASETS_USED = ["gsm8k"]` in the config cell. To cut further, drop `few_shot`
from `CONDITIONS`; questions 1 and 2 do not need it.

Needs ~21 GB of downloads: `Qwen2-1.5B-Instruct`, `Qwen2.5-1.5B-Instruct`, `Qwen2.5-7B-Instruct`.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs.
# ---------------------------------------------------------------------------
N_ITEMS    = 20
FEW_SHOT_K = 2
GEN_CAP    = 1024        # the math checkpoints carry a 4096-token context in total
SEED       = 123
DATASETS_USED = ["math500_l3", "gsm8k"]
CONDITIONS    = ["forced_blurt", "bare", "zero_shot_cot", "few_shot"]

# Matched on generation. The 1.5B rows are matched on SIZE too, so general against math
# differs in training and nothing else -- that pair is the clean specialisation control.
# The 3B row is deliberately unmatched: it is the "smaller specialist beats bigger
# generalist" comparison, and its counterpart is the 1.5B math model of its generation.
PAIRS = [
    dict(gen="2",   size="1.5B", general="Qwen/Qwen2-1.5B-Instruct",
                                 math="Qwen/Qwen2-Math-1.5B-Instruct",   matched_size=True),
    dict(gen="2",   size="7B",   general="Qwen/Qwen2-7B-Instruct",
                                 math="Qwen/Qwen2-Math-7B-Instruct",     matched_size=True),
    dict(gen="2.5", size="1.5B", general="Qwen/Qwen2.5-1.5B-Instruct",
                                 math="Qwen/Qwen2.5-Math-1.5B-Instruct", matched_size=True),
    dict(gen="2.5", size="7B",   general="Qwen/Qwen2.5-7B-Instruct",
                                 math="Qwen/Qwen2.5-Math-7B-Instruct",   matched_size=True),
    dict(gen="2.5", size="3B",   general="Qwen/Qwen2.5-3B-Instruct",
                                 math="Qwen/Qwen2.5-Math-1.5B-Instruct", matched_size=False),
]

ROSTER, _seen = [], set()
for p in PAIRS:
    for arm in ("general", "math"):
        mid = p[arm]
        if mid in _seen:
            continue
        _seen.add(mid)
        size = next(s for s in ("1.5B", "3B", "7B") if s in mid.split("/")[-1])
        ROSTER.append(dict(id=mid, arm=arm, gen=p["gen"], size=size,
                           quant="8bit" if size == "7B" else "bf16", no_think=False))
for m in ROSTER:
    m["tag"] = m["id"].split("/")[-1] + ("" if m["quant"] == "bf16" else "[8bit]")

print(f"{len(ROSTER)} checkpoints x {len(DATASETS_USED)} datasets x {len(CONDITIONS)} conditions "
      f"x {N_ITEMS} items = {len(ROSTER)*len(DATASETS_USED)*len(CONDITIONS)*N_ITEMS} generations\n")
for p in PAIRS:
    print(f"  gen {p['gen']:<4} {p['size']:<5} {p['general'].split('/')[-1]:<26} vs "
          f"{p['math'].split('/')[-1]:<28}{'same size' if p['matched_size'] else 'SMALLER math model'}")

In [ ]:
import json
import random
import re
from datetime import datetime, timezone
from pathlib import Path

import requests
import torch
from tqdm.auto import tqdm

import harness

from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test, render_prompt, extract_final_candidate, grade_answer)

OUT_DIR = Path("out/experiment_2"); OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data"); DATA_DIR.mkdir(parents=True, exist_ok=True)
TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"

mean, median, wilson = harness.mean, harness.median, harness.wilson
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])

## The item sets

Same draw as experiment 1, so the two are directly comparable.

In [ ]:
GSM8K_URL = ("https://raw.githubusercontent.com/openai/grade-school-math/"
             "master/grade_school_math/data/test.jsonl")


def load_gsm8k(path=DATA_DIR / "gsm8k_test.jsonl"):
    """GSM8K's test split. The reference answer is the text after '#### '."""
    if not path.exists():
        path.write_text(requests.get(GSM8K_URL, timeout=60).text, encoding="utf-8")
    rows = []
    for i, line in enumerate(path.read_text(encoding="utf-8").splitlines()):
        if not line.strip():
            continue
        r = json.loads(line)
        body, _, final = r["answer"].rpartition("####")
        rows.append({"unique_id": f"gsm8k/{i}", "problem": r["question"],
                     "answer": final.strip().replace(",", ""),
                     "solution": re.sub(r"<<.*?>>", "", body).strip(),
                     "level": 0, "subject": "grade school"})
    return rows


def draw(rows, n, offset):
    rows = sorted(rows, key=lambda r: r["unique_id"])
    random.Random(SEED + offset).shuffle(rows)
    return rows[:n], rows[n:n + FEW_SHOT_K]


math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")
DATASETS = {}
for name, rows in (("math500_l3", [r for r in math500 if int(r["level"]) == 3]),
                   ("gsm8k", load_gsm8k())):
    if name not in DATASETS_USED:
        continue
    items, shots = draw(rows, N_ITEMS, 0)
    DATASETS[name] = {"items": items, "exemplars": shots,
                      "hash": harness.sha256_of(r["unique_id"] for r in items)}
    assert not ({r["unique_id"] for r in items} & {r["unique_id"] for r in shots})
    print(f"{name:<12} {len(items)} scored + {len(shots)} exemplars   "
          f"sha256 {DATASETS[name]['hash'][:12]}   of {len(rows)} available")

print("\nan exemplar, as it will be prepended:\n")
e = DATASETS["gsm8k"]["exemplars"][0]
print(render_prompt(e["question"] if "question" in e else e["problem"]))
print(e["solution"][:300])

## Grading

Inherited from ch03, unchanged, and serving both datasets.

In [ ]:
CHECKS = [
    (r"The answer is \boxed{18}.",           "18",   True),    # MATH shape
    (r"She makes $18 every day.",            "18",   True),    # GSM8K shape, no box
    (r"\boxed{\frac{1}{2}}",                 "0.5",  True),
    (r"\boxed{1,024}",                       "1024", True),
    (r"I am not sure.",                      "18",   False),
    (r"Step 1: 16-3=13. Step 2: 13-4=9. So \boxed{18}.", "18", True),
]
rows = [(t, g, e, bool(grade_answer(extract_final_candidate(t), g))) for t, g, e in CHECKS]
bad = [r for r in rows if r[2] != r[3]]
if bad:
    for t, g, e, got in rows:
        print(f"{'ok ' if e == got else 'FAIL'} {extract_final_candidate(t)!r:<28} vs {g!r:<8} {got}")
    raise AssertionError(f"{len(bad)} grader check(s) behaved unexpectedly")
print(f"grader preflight: {len(rows)}/{len(rows)} as expected, both dataset shapes")

## The four conditions

| condition | what is added after ch03's stem |
|---|---|
| `forced_blurt` | "Give only the final answer. No explanation, no working, no other text." |
| `bare` | nothing |
| `zero_shot_cot` | "Explain step by step." |
| `few_shot` | k reference solutions prepended |

`bare` and `zero_shot_cot` are separated because experiment 1 showed bare already reasons: the
question is whether *asking* adds anything on top of the default.

In [ ]:
BLURT_SUFFIX = "\n\nGive only the final answer. No explanation, no working, no other text."
COT_SUFFIX   = "\n\nExplain step by step."


def build_prompt(condition, question, exemplars):
    stem = render_prompt(question)
    if condition == "bare":
        return stem
    if condition == "forced_blurt":
        return stem + BLURT_SUFFIX
    if condition == "zero_shot_cot":
        return stem + COT_SUFFIX
    shots = "".join(render_prompt(e.get("question", e.get("problem"))) + "\n"
                    + e["solution"].strip() + "\n\n" for e in exemplars[:FEW_SHOT_K])
    return shots + stem


MIN_STEP_CHARS, MIN_CHAIN_STEPS = 12, 2


def _raw_spans(text):
    for pattern in (r"\n\s*\n", r"\n", r"(?<=[.!?])\s+"):
        spans, pos = [], 0
        for m in re.finditer(pattern, text):
            if m.start() > pos:
                spans.append((pos, m.start()))
            pos = m.end()
        if pos < len(text):
            spans.append((pos, len(text)))
        spans = [(a, b) for a, b in spans if text[a:b].strip()]
        if len(spans) > 1:
            return spans
    return [(0, len(text))] if text.strip() else []


def n_steps(text):
    # An empty think block is not a step. Experiment 1 counted it and inflated every
    # Qwen3 row by one.
    text = re.sub(r"^\s*<think>\s*</think>\s*", "", text.strip())
    raw, out = _raw_spans(text), []
    for k, (a, b) in enumerate(raw):
        piece = text[a:b].strip()
        mergeable = (len(piece) < MIN_STEP_CHARS and not re.search(r"\d", piece)
                     and k != len(raw) - 1 and "\\boxed" not in piece)
        if out and mergeable:
            out[-1] = (out[-1][0], b)
        else:
            out.append((a, b))
    return len(out)


def tokens_to_answer(tokens):
    run = ""
    for i, t in enumerate(tokens):
        run += t
        if "\\boxed{" in run:
            return i + 1
    return None


COMPLIES_UNDER = 40      # a response this short stated an answer and nothing else
print("conditions:", CONDITIONS)

## The sweep

One checkpoint at a time, loaded, swept, freed. The cap is clipped to what each
checkpoint's context leaves after the prompt.

In [ ]:
RUN_ID = f"exp2-n{N_ITEMS}-k{FEW_SHOT_K}-cap{GEN_CAP}-{SEED}"
STORE = harness.TraceStore(TRACES_PATH, RUN_ID, ("tag", "dataset", "condition", "problem_id"))
STORE.report()

device = get_device()
DTYPE = harness.pick_dtype(device)


def sweep_model(entry):
    loaded = harness.load_model(entry["id"], device=device, dtype=DTYPE,
                                load_in_8bit=entry["quant"] == "8bit")
    model, tok = loaded["model"], loaded["tokenizer"]
    ctx = getattr(model.config, "max_position_embeddings", 32768)
    print(f"      context {ctx} tokens")

    for ds_name, ds in DATASETS.items():
        for cond in CONDITIONS:
            todo = [r for r in ds["items"]
                    if STORE.get(entry["tag"], ds_name, cond, r["unique_id"]) is None]
            bar = tqdm(ds["items"], desc=f"{entry['tag'][:22]:<22} {ds_name:<10} {cond:<9}",
                       unit="item")
            for row in bar:
                if STORE.get(entry["tag"], ds_name, cond, row["unique_id"]) is not None:
                    continue
                text = build_prompt(cond, row["problem"], ds["exemplars"])
                if entry["no_think"]:
                    text += " /no_think"     # Qwen3's in-prompt route to thinking off
                chat = tok.apply_chat_template([{"role": "user", "content": text}],
                                               tokenize=False, add_generation_prompt=True)
                n_prompt = len(tok(chat, add_special_tokens=False).input_ids)
                cap = max(64, min(GEN_CAP, ctx - n_prompt - 32))
                g = harness.complete(chat, model, tok, stop=["\nQuestion:"], max_new_tokens=cap)
                got = extract_final_candidate(g["text"])
                STORE.append({
                    "tag": entry["tag"], "model_id": entry["id"], "arm": entry["arm"],
                    "generation": entry["gen"], "quant": entry["quant"], "size": entry["size"],
                    "dataset": ds_name, "condition": cond,
                    "problem_id": row["unique_id"], "question": row["problem"],
                    "ground_truth": row["answer"], "trace": g["text"], "tokens": g["tokens"],
                    "prompt_tokens": n_prompt, "cap": cap, "truncated": g["truncated"],
                    "tokens_generated": g["tokens_generated"],
                    "steps": n_steps(g["text"]),
                    "tokens_to_answer": tokens_to_answer(g["tokens"]),
                    "extracted_answer": got,
                    "correct": bool(grade_answer(got, row["answer"])),
                })
    del model, loaded
    harness.free_memory()


for entry in ROSTER:
    sweep_model(entry)
print("\nscreen complete")

## The grid

In [ ]:
records = [json.loads(l) for l in TRACES_PATH.open(encoding="utf-8") if l.strip()]
records = [r for r in records if r["run_id"] == RUN_ID]


def cell(tag, ds, cond):
    rs = [r for r in records if r["tag"] == tag and r["dataset"] == ds and r["condition"] == cond]
    if not rs:
        return None
    k = sum(r["correct"] for r in rs)
    return {"tag": tag, "dataset": ds, "condition": cond, "n": len(rs),
            "accuracy": k / len(rs), "accuracy_ci": wilson(k, len(rs)),
            "chain_share": mean([r["steps"] >= MIN_CHAIN_STEPS for r in rs]),
            "complies": mean([r["tokens_generated"] <= COMPLIES_UNDER for r in rs]),
            "median_tokens": median([r["tokens_generated"] for r in rs]),
            "median_steps": median([r["steps"] for r in rs])}


TABLE = [c for e in ROSTER for ds in DATASETS for cond in CONDITIONS
         if (c := cell(e["tag"], ds, cond))]

for ds in DATASETS:
    print(f"\n=== {ds} ===")
    print(f"{'model':<30}{'arm':>8}{'condition':<15}{'acc':>6}{'tok':>7}{'steps':>7}{'comply':>8}")
    for e in ROSTER:
        for cond in CONDITIONS:
            c = next((x for x in TABLE if x["tag"] == e["tag"] and x["dataset"] == ds
                      and x["condition"] == cond), None)
            if c:
                print(f"{e['tag'][:29]:<30}{e['arm']:>8}{cond:<15}{c['accuracy']:>6.0%}"
                      f"{c['median_tokens']:>7.0f}{c['median_steps']:>7.0f}{c['complies']:>8.0%}")

## Question 1 — what suppression costs

Same checkpoint, same items, forbidden to reason against asked to reason. A model that ignores the
instruction shows a compliance near zero and a cost near zero, which is itself the finding.

In [ ]:
Q1 = []
for e in ROSTER:
    for ds in DATASETS:
        fb = next((c for c in TABLE if c["tag"] == e["tag"] and c["dataset"] == ds
                   and c["condition"] == "forced_blurt"), None)
        ct = next((c for c in TABLE if c["tag"] == e["tag"] and c["dataset"] == ds
                   and c["condition"] == "zero_shot_cot"), None)
        ba = next((c for c in TABLE if c["tag"] == e["tag"] and c["dataset"] == ds
                   and c["condition"] == "bare"), None)
        if not (fb and ct and ba):
            continue
        Q1.append(dict(tag=e["tag"], arm=e["arm"], gen=e["gen"], size=e["size"], dataset=ds,
                       complies=fb["complies"], blurt_acc=fb["accuracy"], bare_acc=ba["accuracy"],
                       cot_acc=ct["accuracy"], blurt_tokens=fb["median_tokens"],
                       cot_tokens=ct["median_tokens"],
                       suppression_cost_pts=(ct["accuracy"] - fb["accuracy"]) * 100,
                       asking_gain_pts=(ct["accuracy"] - ba["accuracy"]) * 100))

print(f"{'model':<30}{'arm':>8}{'ds':<11}{'comply':>8}{'blurt':>7}{'bare':>7}{'cot':>6}"
      f"{'cost':>7}{'ask+':>6}{'tok x':>7}")
for q in Q1:
    mult = q["cot_tokens"] / max(q["blurt_tokens"], 1)
    print(f"{q['tag'][:29]:<30}{q['arm']:>8}{q['dataset']:<11}{q['complies']:>7.0%}"
          f"{q['blurt_acc']:>7.0%}{q['bare_acc']:>7.0%}{q['cot_acc']:>6.0%}"
          f"{q['suppression_cost_pts']:>+7.0f}{q['asking_gain_pts']:>+6.0f}{mult:>6.0f}x")

## Question 2 — what specialisation buys

Each math checkpoint against the general checkpoint it is paired with, evaluated twice: once with
the general model forbidden to reason, once with it asked to. The first is the headline number,
the second is the honest one — if a math model only wins against a suppressed opponent, that is a
much weaker claim and the page must say so.

In [ ]:
Q2 = []
for p in PAIRS:
    # Skip rather than raise: the roster can legitimately be a subset, and a missing
    # checkpoint should cost one row of analysis, not the whole cell.
    gtag = next((e["tag"] for e in ROSTER if e["id"] == p["general"]), None)
    mtag = next((e["tag"] for e in ROSTER if e["id"] == p["math"]), None)
    if gtag is None or mtag is None:
        continue
    for ds in DATASETS:
        m_bare = next((c for c in TABLE if c["tag"] == mtag and c["dataset"] == ds
                       and c["condition"] == "bare"), None)
        if not m_bare:
            continue
        row = dict(gen=p["gen"], size=p["size"], matched_size=p["matched_size"],
                   math=mtag, general=gtag, dataset=ds, math_acc=m_bare["accuracy"])
        for cond, label in (("forced_blurt", "vs_suppressed"), ("zero_shot_cot", "vs_reasoning")):
            g = next((c for c in TABLE if c["tag"] == gtag and c["dataset"] == ds
                      and c["condition"] == cond), None)
            row[label] = None if g is None else (m_bare["accuracy"] - g["accuracy"]) * 100
            row[label + "_general_acc"] = None if g is None else g["accuracy"]
        Q2.append(row)

print(f"{'gen':<5}{'size':<6}{'ds':<11}{'math model':<28}{'math':>6}"
      f"{'vs blurt-7B':>13}{'vs reasoning':>14}")
for r in Q2:
    a = "-" if r["vs_suppressed"] is None else f"{r['vs_suppressed']:+.0f}"
    b = "-" if r["vs_reasoning"] is None else f"{r['vs_reasoning']:+.0f}"
    star = "" if r["matched_size"] else "  <- smaller math model"
    print(f"{r['gen']:<5}{r['size']:<6}{r['dataset']:<11}{r['math'][:27]:<28}"
          f"{r['math_acc']:>6.0%}{a:>13}{b:>14}{star}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(DATASETS), figsize=(11, 5), sharey=True)
axes = [axes] if len(DATASETS) == 1 else list(axes)
COND_INK = {"forced_blurt": "#b23", "bare": "#9aa0ab", "zero_shot_cot": "#3b4a6b",
            "few_shot": "#a2704a"}
for ax, ds in zip(axes, DATASETS):
    tags = [e["tag"] for e in ROSTER]
    y = range(len(tags))
    w = 0.8 / len(CONDITIONS)
    for j, cond in enumerate(CONDITIONS):
        vals = [next((c["accuracy"] * 100 for c in TABLE if c["tag"] == t
                      and c["dataset"] == ds and c["condition"] == cond), 0) for t in tags]
        ax.barh([i + j * w - 0.4 for i in y], vals, height=w, color=COND_INK[cond],
                label=cond if ax is axes[0] else None)
    ax.set_yticks(list(y))
    ax.set_yticklabels([t[:26] for t in tags], fontsize=8)
    ax.set_xlim(0, 100)
    ax.set_xlabel("solved (%)")
    ax.set_title(ds, fontsize=9, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False, fontsize=8, loc="lower right")
fig.suptitle(f"suppressed vs left alone vs asked vs shown  ·  n={N_ITEMS} paired items",
             fontsize=10, x=0.01, ha="left")
fig.tight_layout()
fig.savefig(OUT_DIR / "figure_experiment_2.png", dpi=160, bbox_inches="tight")
plt.show()

## The record

In [ ]:
derived = {
    "manifest": {
        "notebook": "experiment_2_suppression_cost_and_specialisation.ipynb", "run_id": RUN_ID,
        "harness": harness.fingerprint(),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "notebook_git_sha": harness.git_sha(),
        "n_items": N_ITEMS, "few_shot_k": FEW_SHOT_K, "gen_cap": GEN_CAP, "seed": SEED,
        "conditions": CONDITIONS, "pairs": PAIRS, "roster": ROSTER,
        "datasets": {k: {"hash": v["hash"], "n": len(v["items"])} for k, v in DATASETS.items()},
        "versions": {"torch": torch.__version__,
                     "transformers": __import__("transformers").__version__,
                     "device": str(device)},
    },
    "grid": TABLE,
    "q1_suppression": Q1,
    "q2_specialisation": Q2,
    "thresholds": {"complies_under_tokens": COMPLIES_UNDER, "min_chain_steps": MIN_CHAIN_STEPS},
}
DERIVED_PATH.write_text(json.dumps(derived, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"derived.json   {harness.kb(DERIVED_PATH):>9.1f} KB")
print(f"traces.jsonl   {harness.kb(TRACES_PATH):>9.1f} KB")